# SQL query from table names - Continued

In [3]:
pip show openai


Note: you may need to restart the kernel to use updated packages.
Name: openai
Version: 0.28.0
Summary: Python client library for the OpenAI API
Home-page: https://github.com/openai/openai-python
Author: OpenAI
Author-email: support@openai.com
License: 
Location: c:\Users\joshu\.anaconda\Lib\site-packages
Requires: aiohttp, requests, tqdm
Required-by: langchain-openai


In [4]:
!pip install openai

In [5]:
import openai 

In [7]:
pip install --upgrade openai


  Using cached openai-1.60.2-py3-none-any.whl.metadata (27 kB)
Using cached openai-1.60.2-py3-none-any.whl (456 kB)
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0
Note: you may need to restart the kernel to use updated packages.


In [10]:
!pip uninstall openai
!pip install openai


^C


In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

Python-dotenv could not parse statement starting at line 14
Python-dotenv could not parse statement starting at line 19
Python-dotenv could not parse statement starting at line 20
Python-dotenv could not parse statement starting at line 21
Python-dotenv could not parse statement starting at line 22
Python-dotenv could not parse statement starting at line 23
Python-dotenv could not parse statement starting at line 24


## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [7]:
context = [ {'role':'system', 'content':"""
You are a SQL expert. Generate SQL queries based on the provided database schema.

Database Schema:
CREATE TABLE Students (
    student_id INT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    major VARCHAR(50)
);

CREATE TABLE Courses (
    course_id INT PRIMARY KEY,
    course_name VARCHAR(100),
    department VARCHAR(50)
);

CREATE TABLE Enrollments (
    enrollment_id INT PRIMARY KEY,
    student_id INT,
    course_id INT,
    semester VARCHAR(10),
    FOREIGN KEY (student_id) REFERENCES Students(student_id),
    FOREIGN KEY (course_id) REFERENCES Courses(course_id)
);

Sample Data:
Students:
(101, 'Alice', 22, 'Computer Science'),
(102, 'Bob', 21, 'Mathematics'),
(103, 'Charlie', 23, 'Physics');

Courses:
(201, 'Database Systems', 'Computer Science'),
(202, 'Linear Algebra', 'Mathematics'),
(203, 'Quantum Mechanics', 'Physics');

Enrollments:
(301, 101, 201, 'Fall 2024'),
(302, 102, 202, 'Spring 2024'),
(303, 103, 203, 'Fall 2024');

Examples:
User: "Which students are taking the Database Systems course?"
SQL:
SELECT s.name
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id
WHERE c.course_name = 'Database Systems';

User: "Show me all courses in the Mathematics department."
SQL:
SELECT course_name
FROM Courses
WHERE department = 'Mathematics';

User: "How many students are enrolled in each course?"
SQL:
SELECT c.course_name, COUNT(e.student_id) AS student_count
FROM Courses c
LEFT JOIN Enrollments e ON c.course_id = e.course_id
GROUP BY c.course_name;
"""} ]




In [8]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
-- Question 1: Which students are taking the "Database Systems" course?
SELECT s.name
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id
WHERE c.course_name = 'Database Systems';

-- Question 2: List all courses offered by the 'Mathematics' department.
SELECT course_name
FROM Courses
WHERE department = 'Mathematics';

-- Question 3: How many students are enrolled in each course?
SELECT c.course_name, COUNT(e.student_id) AS student_count
FROM Courses c
LEFT JOIN Enrollments e ON c.course_id = e.course_id
GROUP BY c.course_name;

-- Question 4: Get all students and their enrolled courses.
SELECT s.name, c.course_name, e.semester
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id;

-- Question 5: Find the average age of students in each major.
SELECT major, AVG(age) AS avg_age
FROM Students
GROUP BY major;

-- Question 6: Show all students who are NOT enrolled in any courses.
SELECT s.name
FROM Students s
LEFT JOIN Enrollments e ON s.student_id = e.student_id
WHERE e.student_id IS NULL;
"""})

In [9]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [10]:
#new
context_user = context.copy()
print(return_CCRMSQL("""YOUR QUERY HERE""", context_user))

```sql
SELECT s.name
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id
WHERE c.course_name = 'Database Systems';
```


In [12]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("SELECT name FROM Students WHERE age > 20;", old_context_user))

This is your SQL:
```sql
SELECT name FROM Students WHERE age > 20;
```
This SQL statement selects the names of students from the "Students" table where the age is greater than 20. However, it seems like there is no "Students" table in the provided database schema.


In [13]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context_user))

Apologies for the oversight. Here is the SQL query for the question you provided:

```sql
-- Question: Show all students who are NOT enrolled in any courses.
SELECT s.name
FROM Students s
LEFT JOIN Enrollments e ON s.student_id = e.student_id
WHERE e.student_id IS NULL;
``` 

This query will retrieve all students who are not enrolled in any courses.


In [14]:
#old
print(return_CCRMSQL("YOUR QUERY HERE", old_context_user))

This is your SQL:
```sql

SELECT * 
FROM employees;
```

This SQL query selects all data from the "employees" table.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [17]:
context = [
    {'role':'system', 'content':"""
You are an expert SQL generator. Generate simple and efficient SQLite queries based on the following schema.

Database Schema:
CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    major TEXT
);

CREATE TABLE Courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT,
    department TEXT
);

CREATE TABLE Enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER,
    semester TEXT,
    FOREIGN KEY (student_id) REFERENCES Students(student_id),
    FOREIGN KEY (course_id) REFERENCES Courses(course_id)
);

Sample Data:
Students:
(101, 'Alice', 22, 'Computer Science'),
(102, 'Bob', 21, 'Mathematics'),
(103, 'Charlie', 23, 'Physics');

Courses:
(201, 'Database Systems', 'Computer Science'),
(202, 'Linear Algebra', 'Mathematics'),
(203, 'Quantum Mechanics', 'Physics');

Enrollments:
(301, 101, 201, 'Fall 2024'),
(302, 102, 202, 'Spring 2024'),
(303, 103, 203, 'Fall 2024');

-- Answer the following queries using SQLite syntax:
"""}
]


In [18]:

context.append({'role':'system', 'content':"""
-- Question 1: Get all students enrolled in "Database Systems."
SELECT s.name
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id
WHERE c.course_name = 'Database Systems';

-- Question 2: List the number of students per course.
SELECT c.course_name, COUNT(e.student_id) AS student_count
FROM Courses c
LEFT JOIN Enrollments e ON c.course_id = e.course_id
GROUP BY c.course_name;

-- Question 3: Find students who are NOT enrolled in any courses.
SELECT s.name
FROM Students s
LEFT JOIN Enrollments e ON s.student_id = e.student_id
WHERE e.student_id IS NULL;
"""})


In [20]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context))

-- Question 4: Get the names of students who are enrolled in a course in the 'Computer Science' department.
SELECT s.name
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id
WHERE c.department = 'Computer Science';


In [21]:
context = [
    {'role':'system', 'content':"""
You are a skilled SQL assistant. Your job is to generate optimized SQLite queries for a movie database.

Database Schema:
CREATE TABLE Movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT,
    release_year INTEGER,
    genre TEXT
);

CREATE TABLE Actors (
    actor_id INTEGER PRIMARY KEY,
    name TEXT,
    birth_year INTEGER
);

CREATE TABLE Cast (
    cast_id INTEGER PRIMARY KEY,
    movie_id INTEGER,
    actor_id INTEGER,
    role TEXT,
    FOREIGN KEY (movie_id) REFERENCES Movies(movie_id),
    FOREIGN KEY (actor_id) REFERENCES Actors(actor_id)
);

Sample Data:
Movies:
(1, 'Inception', 2010, 'Sci-Fi'),
(2, 'The Matrix', 1999, 'Sci-Fi'),
(3, 'Titanic', 1997, 'Romance');

Actors:
(101, 'Leonardo DiCaprio', 1974),
(102, 'Keanu Reeves', 1964),
(103, 'Kate Winslet', 1975);

Cast:
(201, 1, 101, 'Cobb'),
(202, 2, 102, 'Neo'),
(203, 3, 103, 'Rose');

-- Answer the following queries using SQLite:
"""}
]


In [22]:

context.append({'role':'system', 'content':"""
-- Question 1: Find all movies starring Leonardo DiCaprio.
SELECT m.title
FROM Movies m
JOIN Cast c ON m.movie_id = c.movie_id
JOIN Actors """ })


In [23]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context))

-- Answer 1:
SELECT m.title
FROM Movies m
JOIN Cast c ON m.movie_id = c.movie_id
JOIN Actors a ON c.actor_id = a.actor_id
WHERE a.name = 'Leonardo DiCaprio';


In [24]:
context = [
    {'role':'system', 'content':"""
You are a SQL query generator for a hospital database. Write SQLite-compatible queries.

Database Schema:
CREATE TABLE Patients (
    patient_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    gender TEXT
);

CREATE TABLE Doctors (
    doctor_id INTEGER PRIMARY KEY,
    name TEXT,
    specialty TEXT
);

CREATE TABLE Appointments (
    appointment_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    doctor_id INTEGER,
    appointment_date TEXT,
    FOREIGN KEY (patient_id) REFERENCES Patients(patient_id),
    FOREIGN KEY (doctor_id) REFERENCES Doctors(doctor_id)
);

Sample Data:
Patients:
(1, 'John Doe', 45, 'Male'),
(2, 'Jane Smith', 32, 'Female'),
(3, 'Robert Brown', 28, 'Male');

Doctors:
(101, 'Dr. Emily White', 'Cardiology'),
(102, 'Dr. Mark Green', 'Orthopedics');

Appointments:
(201, 1, 101, '2025-02-01'),
(202, 2, 102, '2025-02-05');

-- Answer the following queries using SQLite:
"""}
]


In [25]:

context.append({'role':'system', 'content':"""
-- Question 1: List all patients who have an appointment with Dr. Emily White.
SELECT p.name
FROM Patients p
JOIN Appointments a ON p.patient_id = a.patient_id
JOIN Doctors d ON a.doctor_id = d.doctor_id
WHERE d.name = 'Dr. Emily White';

-- Question 2: Count the number of appointments per doctor.
SELECT d.name, COUNT(a.appointment_id) AS appointment_count
FROM Doctors d
LEFT JOIN Appointments a ON d.doctor_id = a.doctor_id
GROUP BY d.name;

-- Question 3: Find patients who do not have any scheduled appointments.
SELECT p.name
FROM Patients p
LEFT JOIN Appointments a ON p.patient_id = a.patient_id
WHERE a.patient_id IS NULL;
"""})


In [26]:
#new
print(return_CCRMSQL("YOUR QUERY HERE", context))

-- Question: Find the details of appointments scheduled for patient with ID 2.
SELECT a.appointment_id, d.name AS doctor_name, d.specialty, a.appointment_date
FROM Appointments a
JOIN Doctors d ON a.doctor_id = d.doctor_id
WHERE a.patient_id = 2;
